# Module 3.2: Persistent Mounting (`/etc/fstab`) & Access Permissions

## Homelab NAS Development - Step 3: Storage Expansion

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Query drive Unique Universal Identifiers (UUID) for stable device mapping.
2. Understand critical mount flags: `noatime`, `nofail`, and TRIM discard timers.
3. Configure `/etc/fstab` for fail-safe, persistent mounting on system boot.
4. Establish user, group, and directory ownership for Docker container data paths.
5. Validate mounting resilience against system reboots.

## 3.2.1 Conceptual Background: Device Naming vs. UUID

In Linux, device nodes like `/dev/sda` and `/dev/sdb` are dynamically assigned at boot time depending on the enumeration sequence of the USB bus. If you plug in a second flash drive or reboot with another USB peripheral attached, your primary SSD might shift from `/dev/sda` to `/dev/sdb`, breaking hardcoded mount paths.

To ensure deterministic mounting across reboots and hardware topology changes, Linux administrators bind mounts by **UUID (Universally Unique Identifier)**—a 128-bit cryptographic identifier stamped permanently into the `ext4` filesystem superblock when the partition was formatted.

## 3.2.2 Step-by-Step Laboratory Guide

### Task 3.2.1: Retrieve the Partition UUID
Query the block device attributes on `/dev/sda1` using `blkid` to inspect its filesystem metadata:
```bash
sudo blkid /dev/sda1
```
Copy the `UUID="xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"` string and record it below for reference in Task 3.2.3:

> ### 📝 User Partition UUID Record
> *Record your partition's `UUID` below:*
> 
> ```markdown
> UUID="YOUR_ACTUAL_UUID_HERE"
> ```

### Task 3.2.2: Create Dedicated Mount Directories
In Linux, external drives are not assigned drive letters (like `D:` in Windows); instead, they are mounted onto an empty directory inside the unified filesystem tree (conventionally under `/mnt` for permanent storage volumes).

Create the mount point directory using `mkdir -p` (`-p` ensures parent directories are created if missing and suppresses errors if the folder already exists):
```bash
sudo mkdir -p /mnt/homelab_storage
```

### Task 3.2.3: Configure `/etc/fstab`

The `/etc/fstab` (File Systems Table) file is read by `systemd` during boot (and whenever `mount -a` is invoked) to automatically mount block devices to their designated directory paths.

1. **Create a safety backup** of your current `/etc/fstab` before making changes:
```bash
sudo cp /etc/fstab /etc/fstab.backup
```

2. **Open `/etc/fstab`** in `nano`:
```bash
sudo nano /etc/fstab
```

3. **Append the persistent mount entry** at the bottom of the file (replace `YOUR_ACTUAL_UUID_HERE` with your UUID from Task 3.2.1):
```text
UUID=YOUR_ACTUAL_UUID_HERE   /mnt/homelab_storage   ext4   defaults,noatime,nofail   0   2
```

#### Understanding the 6 `/etc/fstab` Fields & Critical Mount Flags
Each line in `/etc/fstab` consists of six whitespace-separated columns that define how the kernel mounts and checks the filesystem:

1. **`UUID=YOUR_ACTUAL_UUID_HERE` (File System Specifier):** Identifies the exact partition by its permanent 128-bit UUID rather than a volatile device node like `/dev/sda1`. (Modern `libmount` on Debian/Raspberry Pi OS accepts both unquoted `UUID=...` and `blkid`-style quoted `UUID="..."`, though unquoted is the traditional convention).
2. **`/mnt/homelab_storage` (Mount Point):** The target directory in the root filesystem tree where the SSD's contents will be attached and accessed.
3. **`ext4` (File System Type):** Explicitly instructs the kernel to use the Fourth Extended Filesystem driver created in Module 3.1.
4. **`defaults,noatime,nofail` (Critical Mount Flags):** Comma-separated behavioral flags (with no spaces between commas):
   - **`defaults`**: Enables the standard baseline mount options (`rw` read-write, `suid`, `dev`, `exec`, `auto` mount at boot, `nouser`, and `async` I/O).
   - **`noatime` (Disable Access Time Updates)**: By default, Linux performs a disk write to update a file's access timestamp (`atime`) every single time that file is read. On a photo and media server hosting thousands of images and thumbnails, browsing a gallery triggers thousands of unnecessary metadata write operations. Specifying `noatime` disables read-timestamp updates, significantly boosting I/O read speed and extending NVMe SSD flash longevity.
   - **`nofail` (Boot Fail-Safe)**: By default, if a drive listed in `/etc/fstab` is unplugged or slow to power up during boot, `systemd` halts the boot sequence and drops into an interactive emergency maintenance shell—rendering a headless SSH server unreachable. With `nofail`, the Raspberry Pi logs a warning and continues booting normally even if the external SSD is disconnected.
5. **`0` (Dump Backup Flag):** A legacy flag used by the `dump` backup utility. Setting `0` disables obsolete `dump` checks.
6. **`2` (Filesystem Check Order / `fsck` Pass):** Controls when `fsck` (File System Consistency Check) inspects the drive for corruption at boot. The root OS drive (`/`) uses `1` (checked first), while secondary data drives should be set to `2` so they are verified after the root filesystem is online (`0` disables boot-time integrity checks).

### Task 3.2.4: Test Mount, Configure Access Permissions & Verify SSD TRIM

#### 1. Test `/etc/fstab` Without Rebooting
Always validate `/etc/fstab` syntax immediately after editing. If there is a typo in `/etc/fstab`, catching it now lets you fix it in your active terminal session before a reboot:
- **`sudo mount -a`**: Instructs the kernel to mount all filesystems declared in `/etc/fstab` that are not yet mounted. If this command returns silently with no errors, your syntax is valid.
- **`sudo systemctl daemon-reload`**: Refreshes `systemd`'s dynamically generated `.mount` unit files so the service manager recognizes your updated `/etc/fstab` configuration.
- **`df -h /mnt/homelab_storage`**: Reports **Disk Free** space for the mount point in human-readable units (`-h`, displaying GB/TB), confirming that `/dev/sda1` is actively mounted on `/mnt/homelab_storage`.

```bash
sudo mount -a
sudo systemctl daemon-reload
df -h /mnt/homelab_storage
```

#### 2. Configure Directory Ownership & Access Permissions
When `mkfs.ext4` formats a partition, the root directory of that new filesystem is owned exclusively by `root:root` with `755` permissions. Standard users and Docker containers running under your administrator account (`pi`) will encounter `Permission denied` errors when attempting to create folders or upload media unless ownership is delegated:
- **`sudo chown -R pi:pi /mnt/homelab_storage`**: Changes ownership (`chown`) recursively (`-R`) across `/mnt/homelab_storage` so that both the owning **user** and owning **group** are set to `pi` (`user:group`).
- **`sudo chmod 775 /mnt/homelab_storage`**: Sets directory permissions using octal notation (`775` = `rwxrwxr-x`):
  - **Owner (`7` = `rwx`)**: User `pi` can read, write, and traverse directories.
  - **Group (`7` = `rwx`)**: Any service or container assigned to the `pi` group has full read/write/execute access.
  - **Others (`5` = `r-x`)**: Unprivileged system processes can read and traverse directories if needed, but cannot modify or delete files.

```bash
sudo chown -R pi:pi /mnt/homelab_storage
sudo chmod 775 /mnt/homelab_storage
```

#### 3. Verify Background SSD TRIM Support (`fstrim.timer`)
Unlike mechanical hard drives, NVMe Solid State Drives cannot overwrite existing flash cells directly—blocks must be erased before they can be rewritten. **TRIM** (`discard`) informs the SSD's internal controller which blocks contain deleted files and are no longer in use, allowing the drive's garbage collection and wear-leveling algorithms to clean them proactively during idle periods.

Rather than mounting with continuous inline `discard` (which adds latency to every file deletion), Debian and Raspberry Pi OS use a scheduled `systemd` timer (**`fstrim.timer`**) that runs `fstrim` across all mounted SSDs once a week in the background.

Verify that the weekly TRIM timer is enabled and `active (waiting)`:
```bash
sudo systemctl status fstrim.timer
```

### 🖼️ Task 3.2.4 Verification: Persistent Mount Verification (`df -h` and `fstab`)

**Verification Objective:** Attach a screenshot showing the output of `cat /etc/fstab` and `df -h /mnt/homelab_storage` confirming the drive is mounted.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Partition `UUID` retrieved via `blkid` and recorded.
- [ ] Dedicated mount point `/mnt/homelab_storage` created and configured in `/etc/fstab` with `defaults,noatime,nofail`.
- [ ] Persistent mount verified with `mount -a` and `df -h`, directory permissions set (`pi:pi`, `775`), and `fstrim.timer` confirmed active.
- [ ] Verification screenshot captured and attached directly above.

Proceed to the next module in sequence once this checkpoint is satisfied.